setup

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("../data/raw/application_train.csv")

print(df.shape)

(307511, 122)


In [4]:
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(
    365243,
    np.nan
)

In [5]:
def create_features(df):
    
    df = df.copy()
    
    # Age and employment
    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25
    df["EMPLOYMENT_YEARS"] = -df["DAYS_EMPLOYED"] / 365.25
    
    # Financial ratios
    df["INCOME_TO_CREDIT_RATIO"] = (
        df["AMT_INCOME_TOTAL"] /
        df["AMT_CREDIT"]
    )
    
    df["CREDIT_TO_INCOME_RATIO"] = (
        df["AMT_CREDIT"] /
        df["AMT_INCOME_TOTAL"]
    )
    
    df["ANNUITY_TO_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] /
        df["AMT_INCOME_TOTAL"]
    )
    
    df["CREDIT_TO_ANNUITY_RATIO"] = (
        df["AMT_CREDIT"] /
        df["AMT_ANNUITY"]
    )
    
    df["CREDIT_TO_GOODS_RATIO"] = (
        df["AMT_CREDIT"] /
        df["AMT_GOODS_PRICE"]
    )
    
    df["INCOME_PER_PERSON"] = (
        df["AMT_INCOME_TOTAL"] /
        df["CNT_FAM_MEMBERS"]
    )
    
    df["CHILDREN_RATIO"] = (
        df["CNT_CHILDREN"] /
        df["CNT_FAM_MEMBERS"]
    )
    
    df["EMPLOYMENT_TO_AGE_RATIO"] = (
        df["EMPLOYMENT_YEARS"] /
        df["AGE_YEARS"]
    )
    
    # Age groups
    df["AGE_GROUP"] = pd.cut(
        df["AGE_YEARS"],
        bins=[0, 25, 35, 45, 55, 65, 100],
        labels=[
            "18-25",
            "26-35",
            "36-45",
            "46-55",
            "56-65",
            "65+"
        ]
    )
    
    # Employment groups
    df["EMPLOYMENT_GROUP"] = pd.cut(
        df["EMPLOYMENT_YEARS"],
        bins=[-1, 1, 3, 5, 10, 20, 100],
        labels=[
            "<1 Year",
            "1-3 Years",
            "3-5 Years",
            "5-10 Years",
            "10-20 Years",
            "20+ Years"
        ]
    )
    
    # Document aggregate
    document_columns = [
        col for col in df.columns
        if col.startswith("FLAG_DOCUMENT")
    ]
    
    df["TOTAL_DOCUMENTS_SUBMITTED"] = (
        df[document_columns].sum(axis=1)
    )
    
    # External source aggregates
    external_sources = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]
    
    df["EXT_SOURCE_MEAN"] = (
        df[external_sources].mean(axis=1)
    )
    
    df["EXT_SOURCE_MAX"] = (
        df[external_sources].max(axis=1)
    )
    
    df["EXT_SOURCE_MIN"] = (
        df[external_sources].min(axis=1)
    )
    
    df["EXT_SOURCE_STD"] = (
        df[external_sources].std(axis=1)
    )
    
    df["EXT_SOURCE_COUNT"] = (
        df[external_sources]
        .notna()
        .sum(axis=1)
    )
    
    # Contact aggregate
    contact_columns = [
        "FLAG_MOBIL",
        "FLAG_EMP_PHONE",
        "FLAG_WORK_PHONE",
        "FLAG_CONT_MOBILE",
        "FLAG_PHONE",
        "FLAG_EMAIL"
    ]
    
    df["TOTAL_CONTACT_FLAGS"] = (
        df[contact_columns].sum(axis=1)
    )
    
    # Replace infinity created by ratios
    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )
    
    return df

In [6]:
df = create_features(df)

print(df.shape)

(307511, 141)


In [7]:
X = df.drop(
    columns=[
        "TARGET",
        "SK_ID_CURR"
    ]
)

y = df["TARGET"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 121
Categorical: 18


C:\Users\Charan\AppData\Local\Temp\ipykernel_16180\302416908.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [10]:
numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

In [11]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [13]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = (
    negative_count / positive_count
)

print("Scale Positive Weight:", scale_pos_weight)

Scale Positive Weight: 11.38710976837865


In [14]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    
    subsample=0.8,
    colsample_bytree=0.8,
    
    scale_pos_weight=scale_pos_weight,
    
    random_state=42,
    n_jobs=-1
)

In [15]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xgb_model)
    ]
)

In [16]:
xgb_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](139,)","['NAME_CONTRACT_TYPE','CODE_GENDER','FLAG_OWN_CAR',...,'EXT_SOURCE_STD', 'EXT_SOURCE_COUNT','TOTAL_CONTACT_FLAGS']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,139
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By

In [17]:
y_pred = xgb_pipeline.predict(X_test)

y_prob = xgb_pipeline.predict_proba(
    X_test
)[:, 1]

In [18]:
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       0.96      0.74      0.84     56538
           1       0.18      0.66      0.28      4965

    accuracy                           0.73     61503
   macro avg       0.57      0.70      0.56     61503
weighted avg       0.90      0.73      0.79     61503



In [19]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

In [20]:
param_distributions = {
    
    "model__n_estimators": [
        200,
        300,
        500
    ],
    
    "model__max_depth": [
        3,
        4,
        5,
        6,
        8
    ],
    
    "model__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.1
    ],
    
    "model__subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],
    
    "model__colsample_bytree": [
        0.6,
        0.8,
        1.0
    ],
    
    "model__min_child_weight": [
        1,
        3,
        5,
        10
    ],
    
    "model__gamma": [
        0,
        0.1,
        0.3
    ]
}

In [21]:
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    
    param_distributions=param_distributions,
    
    n_iter=15,
    
    scoring="roc_auc",
    
    cv=cv,
    
    verbose=2,
    
    random_state=42,
    
    n_jobs=-1,
    
    return_train_score=True
)

In [ ]:
random_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


In [ ]:
random_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best Parameters:"
)

random_search.best_params_

In [ ]:
best_xgb_model = random_search.best_estimator_

In [ ]:
best_y_pred = best_xgb_model.predict(
    X_test
)

best_y_prob = best_xgb_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
final_roc_auc = roc_auc_score(
    y_test,
    best_y_prob
)

final_pr_auc = average_precision_score(
    y_test,
    best_y_prob
)

print("Final Test ROC-AUC:", final_roc_auc)

print("Final Test PR-AUC:", final_pr_auc)

In [ ]:
cm = confusion_matrix(
    y_test,
    best_y_pred
)

ConfusionMatrixDisplay(
    confusion_matrix=cm
).plot()

plt.title(
    "XGBoost Confusion Matrix"
)

plt.show()

In [ ]:
final_comparison = pd.DataFrame([
    
    {
        "Model": "Logistic Regression",
        "ROC-AUC": logistic_results["ROC-AUC"],
        "PR-AUC": logistic_results["PR-AUC"]
    },
    
    {
        "Model": "Random Forest",
        "ROC-AUC": random_forest_results["ROC-AUC"],
        "PR-AUC": random_forest_results["PR-AUC"]
    },
    
    {
        "Model": "Logistic Regression + FE",
        "ROC-AUC": engineered_logistic_results["ROC-AUC"],
        "PR-AUC": engineered_logistic_results["PR-AUC"]
    },
    
    {
        "Model": "Random Forest + FE",
        "ROC-AUC": engineered_rf_results["ROC-AUC"],
        "PR-AUC": engineered_rf_results["PR-AUC"]
    },
    
    {
        "Model": "Tuned XGBoost",
        "ROC-AUC": final_roc_auc,
        "PR-AUC": final_pr_auc
    }
])

final_comparison.sort_values(
    by="ROC-AUC",
    ascending=False
).round(4)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)

results = []

for threshold in thresholds:
    
    y_pred = (y_prob >= threshold).astype(int)
    
    results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(results)
threshold_df.head()

In [ ]:
best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

FINAL_THRESHOLD = best_row["threshold"]

print("Final Threshold:", FINAL_THRESHOLD)
print(best_row)

In [ ]:
final_predictions = (
    y_prob >= FINAL_THRESHOLD
).astype(int)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(
    classification_report(
        y_test,
        final_predictions
    )
)

print(
    confusion_matrix(
        y_test,
        final_predictions
    )
)